# **DeBERTa**

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1" # "0" o "1"

In [2]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [3]:
from utils import *

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)

import optuna
from functools import partial
from sklearn.utils import shuffle

/home/n.emmolo/miniconda3/envs/env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-11-19 15:10:46.298198: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-19 15:10:46.365715: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-19 15:10:47.710800: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may 

In [4]:
# Ignore warnings for cleaner output
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# ----------------------
# Tokenization functions
# ----------------------

def tokenize_function(examples, tokenizer, max_len=128):
    """
    Tokenizes the input examples using the provided tokenizer.

    Args:
        examples (dict): A dictionary containing the text data to be tokenized.
        tokenizer (AutoTokenizer): The tokenizer to use for tokenization.
        max_len (int): Maximum length for padding/truncation.

    Returns:
        dict: Tokenized inputs with padding and truncation applied.
    """

    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=max_len,
    )


def tokenize_datasets(datasets, tokenizer):
    """
    Tokenizes multiple datasets using the provided tokenizer.

    Args:
        datasets (dict): A dictionary where keys are dataset names and values are dictionaries with 'train', 'val', and 'test' splits.
        tokenizer (AutoTokenizer): The tokenizer to use for tokenization.

    Returns:
        dict: A dictionary with the same keys as input datasets, but with tokenized datasets.
    """

    tokenized_datasets = {}
    
    for name, data in datasets.items():
        print(f"\n=== Tokenizing dataset: {name} ===")

        train_dataset = Dataset.from_dict({"text": data["train"][0], "label": data["train"][1].astype(int)})
        val_dataset = Dataset.from_dict({"text": data["val"][0], "label": data["val"][1].astype(int)})
        test_dataset = Dataset.from_dict({"text": data["test"][0], "label": data["test"][1].astype(int)})

        train_tokenized = train_dataset.map(lambda x: tokenize_function(x, tokenizer), batched=True)
        val_tokenized = val_dataset.map(lambda x: tokenize_function(x, tokenizer), batched=True)
        test_tokenized = test_dataset.map(lambda x: tokenize_function(x, tokenizer), batched=True)

        tokenized_datasets[name] = {
            "train": (train_tokenized, np.array(train_dataset["label"])),
            "val": (val_tokenized, np.array(val_dataset["label"])),
            "test": (test_tokenized, np.array(test_dataset["label"]))
        }

    return tokenized_datasets

In [ ]:
# --------------------
# Build model function
# --------------------

def build_model(learning_rate=2e-5, weight_decay=0.1, epochs=1):
    """
    Builds a DeBERTa model (microsoft/deberta-base) for sequence classification.

    Args:
        learning_rate (float): Learning rate for the optimizer.
        weight_decay (float): Weight decay for AdamW optimizer.
        epochs (int): Number of training epochs.

    Returns:
        model (AutoModelForSequenceClassification): HuggingFace DeBERTa model.
        train_args (TrainingArguments): Default training configuration.
    """

    model = AutoModelForSequenceClassification.from_pretrained("microsoft/deberta-base", num_labels=2)

    """train_args = TrainingArguments(
        output_dir="./deberta_finetune_output",
        eval_strategy="epoch",
        save_strategy="no",
        learning_rate=learning_rate,
        weight_decay=weight_decay,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=3,
        logging_dir="./logs",
        load_best_model_at_end=False,
        logging_steps=50,
        seed=42,
    )"""

    train_args = TrainingArguments(
        output_dir="./optuna_tmp",
        num_train_epochs=epochs,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        learning_rate=learning_rate,
        weight_decay=weight_decay,
        optim="adamw_torch_fused", # fast

        save_strategy="no",
        eval_strategy="no",
        logging_strategy="no",
        report_to="none",
        load_best_model_at_end=False,
    )

    return model, train_args

In [ ]:
# -------------------------
# Optuna objective function
# -------------------------

def objectiveDeBERTa(trial, X_train, y_train, X_val, y_val):
    """
    Optuna objective function for hyperparameter optimization of DeBERTa model.

    Args:
        trial (optuna.trial.Trial): Optuna trial object.
        X_train (Dataset): Tokenized training dataset.
        y_train (np.array): Training labels.
        X_val (Dataset): Tokenized validation dataset.
        y_val (np.array): Validation labels.

    Returns:
        float: Validation F1 score to be maximized.
    """

    learning_rate = trial.suggest_categorical("learning_rate", [5e-5, 3e-5, 2e-5, 1e-5])
    weight_decay = trial.suggest_categorical("weight_decay", [0.0, 0.1, 0.01, 0.001])

    model, train_args = build_model(learning_rate, weight_decay, epochs=1)
    trainer = Trainer(
        model=model,
        args=train_args,
        train_dataset=X_train,
        eval_dataset=X_val,
    )
    
    trainer.train()
    preds = trainer.predict(X_val)
    preds = np.argmax(preds.predictions, axis=1)
    f1 = f1_score(y_val, preds, average="weighted")
    return f1

## VERSION 1: Dataset by Topic

In [8]:
dataset_df = data_by_topic()

# split "politics" in "politics1" and "politics2"
if "politics" in dataset_df:
    dataset_df["politics1"] = dataset_df["politics"].sample(frac=0.5, random_state=42)
    dataset_df["politics2"] = dataset_df["politics"].drop(dataset_df["politics1"].index)
    # drop original "politics" dataset
    del dataset_df["politics"]
    # put "politics1" and "politics2" at the beginning of the dict
    dataset_df = {"politics1": dataset_df["politics1"], "politics2": dataset_df["politics2"], **dataset_df}

print("--- DATASET SIZES BY TOPIC ---")
for topic, df in dataset_df.items():
    print(f"Topic: {topic}, Number of samples: {len(df)}")

#dataset_df = dict(list(dataset_df.items())[5:]) # remove firsts datasets

print("\nSplitting datasets into train/val/test...")
datasets = {topic: split_dataset(df) for topic, df in dataset_df.items()} # split all datasets in train/val/test

tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-base")

print("\nComputing tokenized datasets...")
datasets =  tokenize_datasets(datasets, tokenizer) # tokenize all datasets

--- DATASET SIZES BY TOPIC ---
Topic: politics1, Number of samples: 48738
Topic: politics2, Number of samples: 48738
Topic: general, Number of samples: 12845
Topic: covid, Number of samples: 10559
Topic: syria, Number of samples: 842
Topic: islam, Number of samples: 722
Topic: notredame, Number of samples: 554
Topic: gossip, Number of samples: 500

Splitting datasets into train/val/test...

Computing tokenized datasets...

=== Tokenizing dataset: politics1 ===


Map: 100%|████████████████████████████████████████████| 9748/9748 [00:03<00:00, 2824.79 examples/s]



=== Tokenizing dataset: politics2 ===


Map: 100%|████████████████████████████████████████████| 9748/9748 [00:03<00:00, 3125.19 examples/s]



=== Tokenizing dataset: general ===


...

Map: 100%|██████████████████████████████████████████████| 145/145 [00:00<00:00, 3635.71 examples/s]



=== Tokenizing dataset: notredame ===


Map: 100%|██████████████████████████████████████████████| 111/111 [00:00<00:00, 3565.06 examples/s]



=== Tokenizing dataset: gossip ===


Map: 100%|██████████████████████████████████████████████| 100/100 [00:00<00:00, 1783.46 examples/s]


In [ ]:
# ---------------------------
# Hyperparameter optimization
# ---------------------------

# merge all training and validation data across topics for optimization
X_train = np.concatenate([datasets[topic]['train'][0] for topic in datasets])
y_train = np.concatenate([datasets[topic]['train'][1] for topic in datasets])
X_val = np.concatenate([datasets[topic]['val'][0] for topic in datasets])
y_val = np.concatenate([datasets[topic]['val'][1] for topic in datasets])
# shuffle the combined training and validation data
X_train, y_train = shuffle(X_train, y_train, random_state=42)
X_val, y_val = shuffle(X_val, y_val, random_state=42)

# functool.partial to pass data to the objective function
objective_with_data = partial(objectiveDeBERTa,
                               X_train=X_train,
                               y_train=y_train,
                               X_val=X_val,
                               y_val=y_val)

study = optuna.create_study(direction="maximize") # maximize F1-score
study.optimize(objective_with_data, n_trials=25) # 25 trials; increase for better results

print("Best parameters:", study.best_params)

In [ ]:
# -------------------------------
# Fine-tuning on Dataset by Topic
# -------------------------------

best_params = study.best_params
best_model, train_args = build_model(
    learning_rate=best_params["learning_rate"],
    weight_decay=best_params["weight_decay"],
    epochs=3
)

results_topic = {}
results_full = {}
X_full_test = np.concatenate([datasets[topic]['test'][0] for topic in datasets])
y_full_test = np.concatenate([datasets[topic]['test'][1] for topic in datasets])

# sequential training
for i, (topic, data) in enumerate(datasets.items()):
    print(f"\n=== Phase {i+1}: Training/Fine-tuning on topic: {topic} ===")

    X_train, y_train = data["train"]
    X_val, y_val = data["val"]
    X_test, y_test = data["test"]

    # define trainer
    trainer = Trainer(
        model=best_model,
        args=train_args,
        train_dataset=X_train,
        eval_dataset=X_val,
    )

    # fine-tune on train + val
    trainer.train()

    # evaluate on current dataset
    y_pred = trainer.predict(X_test)
    y_pred = np.argmax(y_pred.predictions, axis=1)
    print(f"\nClassification Report after topic {topic}:")
    print(classification_report(y_test, y_pred))
    print(f"Confusion Matrix after topic {topic}:")
    print(confusion_matrix(y_test, y_pred))
    print(f"Weighted F1-score after topic {topic}:", f1_score(y_test, y_pred, average='weighted'))

    # evaluation on all topics
    print("\n--- Evaluation on all topics ---")
    results_topic[topic] = {}
    for test_topic, test_data in datasets.items(): # for each topic
        X_te, y_te = test_data["test"]
        preds = trainer.predict(X_te)
        preds = np.argmax(preds.predictions, axis=1)
        f1 = f1_score(y_te, preds, average="weighted")
        results_topic[topic][test_topic] = f1
        print(f"Evaluation on topic {test_topic}: Weighted F1 = {f1:.4f}")

    # evaluation on full test set
    print("\n--- Evaluation on full test set ---")
    preds_full = trainer.predict(X_full_test)
    preds_full = np.argmax(preds_full.predictions, axis=1)
    f1_full = f1_score(y_full_test, preds_full, average="weighted")
    results_full[topic] = f1_full
    print(f"Evaluation on full test set: Weighted F1 = {f1_full:.4f}")

In [ ]:
# ---------------
# Results summary
# ---------------

print("\n=== Results Summary ===")
for topic, res in results_topic.items():
    print(f"\nResults after training on topic {topic}:")
    for test_topic, f1 in res.items():
        print(f"  Test on topic {test_topic}: Weighted F1 = {f1:.4f}")

print("\n=== Full Test Set Results Summary ===")
for topic, f1 in results_full.items():
    print(f"After training on topic {topic}: Full Test Set Weighted F1 = {f1:.4f}")

In [ ]:
# ------------------
# Plot on each Topic
# ------------------

# extract topics
topics = list(results_topic.keys())
test_topics = list(next(iter(results_topic.values())).keys())

# make F1-score matrix
f1_matrix = np.array([[results_topic[train_topic][test_topic] for test_topic in test_topics] for train_topic in topics])

plt.figure(figsize=(10, 6))

# plot each test topic line
for i, test_topic in enumerate(test_topics):
    plt.plot(topics, f1_matrix[:, i], marker='o', label=f"Test on {test_topic}")

plt.title("F1-score on each Topic Test Set")
plt.xlabel("Fine-tuning Topic")
plt.ylabel("Weighted F1-score")
plt.ylim(0, 1)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------
# Plot on Full Test Set
# ---------------------

plt.figure(figsize=(6, 4))

# plot F1-score on full test set
plt.plot(topics, [results_full[topic] for topic in topics], marker='o', color='b')

plt.title("F1-score on Full Test Set (Datasets by Topic)")
plt.xlabel("Fine-tuning Topic")
plt.ylabel("Weighted F1-score")
plt.ylim(0, 1)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

## VERSION 2: Dataset by Date

In [ ]:
dataset_df = data_by_date()

print("\n--- DATASET SIZES BY DATE ---")
for date, df in dataset_df.items():
    print(f"Date: {date}, Number of samples: {len(df)}")

# dataset_df = dict(list(dataset_df.items())[:3]) # remove last 3 datasets

print("\nSplitting datasets into train/val/test...")
datasets = {date: split_dataset(df) for date, df in dataset_df.items()} # split all datasets in train/val/test

tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-base")

print("\nComputing tokenized datasets...")
datasets =  tokenize_datasets(datasets, tokenizer) # tokenize all datasets

In [ ]:
# ---------------------------
# Hyperparameter optimization
# ---------------------------

# merge all training and validation data across topics for optimization
X_train = np.concatenate([datasets[topic]['train'][0] for topic in datasets])
y_train = np.concatenate([datasets[topic]['train'][1] for topic in datasets])
X_val = np.concatenate([datasets[topic]['val'][0] for topic in datasets])
y_val = np.concatenate([datasets[topic]['val'][1] for topic in datasets])
# shuffle the combined training and validation data
X_train, y_train = shuffle(X_train, y_train, random_state=42)
X_val, y_val = shuffle(X_val, y_val, random_state=42)

# functool.partial to pass data to the objective function
objective_with_data = partial(objectiveDeBERTa,
                               X_train=X_train,
                               y_train=y_train,
                               X_val=X_val,
                               y_val=y_val)

study = optuna.create_study(direction="maximize") # maximize F1-score
study.optimize(objective_with_data, n_trials=50) # 50 trials for demonstration; increase for better results

print("Best parameters:", study.best_params)

In [ ]:
# ------------------------------
# Fine-tuning on Dataset by Date
# ------------------------------

best_params = study.best_params
best_model, train_args = build_model(
    learning_rate=best_params["learning_rate"],
    weight_decay=best_params["weight_decay"]
)

results_date = {}
results_full = {}
X_full_test = np.concatenate([datasets[date]['test'][0] for date in datasets])
y_full_test = np.concatenate([datasets[date]['test'][1] for date in datasets])

# sequential training
for i, (date, data) in enumerate(datasets.items()):
    print(f"\n=== Phase {i+1}: Training/Fine-tuning on date: {date} ===")
    
    X_train, y_train = data["train"]
    X_val, y_val = data["val"]
    X_test, y_test = data["test"]

    # define trainer
    trainer = Trainer(
        model=best_model,
        args=train_args,
        train_dataset=X_train,
        eval_dataset=X_val,
    )
    
    # fine-tune on train + val
    trainer.train()

    # evaluate on current dataset
    y_pred = trainer.predict(X_test)
    y_pred = np.argmax(y_pred.predictions, axis=1)
    print(f"\nClassification Report after date {date}:")
    print(classification_report(y_test, y_pred))
    print(f"Confusion Matrix after date {date}:")
    print(confusion_matrix(y_test, y_pred))
    print(f"Weighted F1-score after date {date}:", f1_score(y_test, y_pred, average='weighted'))

    # evaluate on all datasets
    print("\n--- Evaluation on all datasets ---")
    results_date[date] = {}
    for test_date, test_data in datasets.items(): # for each date
        X_te, y_te = test_data["test"]
        preds = trainer.predict(X_te)
        preds = np.argmax(preds.predictions, axis=1)
        f1 = f1_score(y_te, preds, average="weighted")
        results_date[date][test_date] = f1
        print(f"Evaluation on date {test_date}: Weighted F1 = {f1:.4f}")

    # evaluation on full test set
    print("\n--- Evaluation on full test set ---")
    preds_full = trainer.predict(X_full_test)
    preds_full = np.argmax(preds_full.predictions, axis=1)
    f1_full = f1_score(y_full_test, preds_full, average="weighted")
    results_full[date] = f1_full
    print(f"Evaluation on full test set: Weighted F1 = {f1_full:.4f}")

In [ ]:
# ---------------
# Results summary
# ---------------

print("\n=== Results Summary ===")
for date, res in results_date.items():
    print(f"\nResults after training on date {date}:")
    for test_date, f1 in res.items():
        print(f"  Test on date {test_date}: Weighted F1 = {f1:.4f}")

print("\n=== Full Test Set Results Summary ===")
for date, f1 in results_full.items():
    print(f"After training on date {date}: Full Test Set Weighted F1 = {f1:.4f}")

In [ ]:
# ------------------
# Plot on each Date
# ------------------

# extract dates
dates = list(results_date.keys())
test_dates = list(next(iter(results_date.values())).keys())

# make F1-score matrix
f1_matrix = np.array([[results_date[train_date][test_date] for test_date in test_dates] for train_date in dates])
plt.figure(figsize=(10, 6))

# plot each test date line
for i, test_date in enumerate(test_dates):
    plt.plot(dates, f1_matrix[:, i], marker='o', label=f"Test on {test_date}")

plt.title("F1-score on each Date Test Set")
plt.xlabel("Fine-tuning Date")
plt.ylabel("Weighted F1-score")
plt.ylim(0, 1)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------
# Plot on Full Test Set
# ---------------------

plt.figure(figsize=(6, 4))

# plot F1-score on full test set
plt.plot(dates, [results_full[date] for date in dates], marker='o', color='b')

plt.title("F1-score on Full Test Set (Datasets by Date)")
plt.xlabel("Fine-tuning Date")
plt.ylabel("Weighted F1-score")
plt.ylim(0, 1)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()